# Business Capability Mapping Tester

This notebook tests capability matching for **selected organizations** (e.g., DGs 23 and 24).

**Features:**
- Load pre-built org and capability graphs (no reprocessing)
- Select specific organizations by code prefix (e.g., "23", "24")
- Run bipartite matching with LLM judge
- Store results and embeddings in graph
- Generate Excel report with match details and justifications

**Outputs:**
- `./output/capability_mapping_report.xlsx` - Detailed Excel report
- `./output/selected_org_matches.json` - JSON match results
- FalkorDB graph: `org_capability_matches`

In [1]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import os
import sys
sys.path.insert(0, '.')

# ═══════════════════════════════════════════════════════════════════════════
# SELECT ORGANIZATIONS TO MAP
# ═══════════════════════════════════════════════════════════════════════════

# Specify org code prefixes to include (e.g., ["23", "24"] for DGs 23 and 24)
# Set to None or [] to map ALL organizations
SELECTED_ORG_PREFIXES = ["10"]

# ═══════════════════════════════════════════════════════════════════════════
# MATCHING MODE (NEW!)
# ═══════════════════════════════════════════════════════════════════════════

# Choose matching strategy:
#   "hybrid"         - Semantic pre-filter + LLM judge (default, balanced)
#   "llm_only"       - Pure LLM matching (most accurate, expensive)
#   "llm_prescreened"- LLM category filter + LLM matching (efficient LLM-only)
#   "semantic_only"  - Embedding similarity only (fastest, least accurate)

MATCHING_MODE = "llm_only"  # ← Change this to switch modes

# ═══════════════════════════════════════════════════════════════════════════
# GRAPH LOADING
# ═══════════════════════════════════════════════════════════════════════════

LOAD_FROM = "local"  # "local" or "falkordb"

# Local paths
ORG_GRAPH_PATH = "./graph_data/organization_graph.json"
CAP_GRAPH_PATH = "./graph_data/capability_graph.json"

# ═══════════════════════════════════════════════════════════════════════════
# FALKORDB SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

USE_REAL_FALKORDB = True
FALKORDB_URL_BASE = "redis://@r-6jissuruar.instance-zeqb0ww84.hc-v8noonp0c.europe-west1.gcp.f2e0a955bb84.cloud:52346"
ORG_GRAPH_NAME = "org_hierarchy"
CAP_GRAPH_NAME = "capability_map"
MATCH_GRAPH_NAME = "org_capability_matches"

# ═══════════════════════════════════════════════════════════════════════════
# MATCHING SETTINGS
# ═══════════════════════════════════════════════════════════════════════════

USE_MOCK_LLM = False           # False = real Claude API
#LLM_MODEL = "claude-sonnet-4-20250514"
LLM_MODEL = "claude-haiku-4-5-20251001"

# Semantic/Hybrid settings
MIN_SEMANTIC_SCORE = 0.20      # Minimum semantic similarity (for hybrid/semantic modes)
TOP_K_CANDIDATES = 10          # Candidates for semantic pre-filter

# LLM settings
MIN_LLM_SCORE = 0.5            # Minimum LLM judge score
LLM_BATCH_SIZE = 15            # Capabilities per LLM batch (for llm_only mode)

# Fallback
ENABLE_HIERARCHICAL_FALLBACK = True  # Try higher cap levels if no leaf match

# Embeddings
USE_MOCK_EMBEDDER = False      # False = real HuggingFace embeddings

# ═══════════════════════════════════════════════════════════════════════════
# OUTPUT
# ═══════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = "./output"
EXCEL_REPORT_PATH = f"{OUTPUT_DIR}/capability_mapping_report.xlsx"
JSON_RESULTS_PATH = f"{OUTPUT_DIR}/selected_org_matches.json"

print("Configuration:")
print(f"  Selected orgs: {SELECTED_ORG_PREFIXES or 'ALL'}")
print(f"  Matching mode: {MATCHING_MODE}")
print(f"  Load from: {LOAD_FROM}")
print(f"  Mock LLM: {USE_MOCK_LLM}")
print(f"  Mock Embedder: {USE_MOCK_EMBEDDER}")
print(f"  Min semantic: {MIN_SEMANTIC_SCORE}")
print(f"  Min LLM: {MIN_LLM_SCORE}")
print(f"  LLM batch size: {LLM_BATCH_SIZE}")
print(f"  Hierarchical fallback: {ENABLE_HIERARCHICAL_FALLBACK}")

Configuration:
  Selected orgs: ['10']
  Matching mode: llm_only
  Load from: local
  Mock LLM: False
  Mock Embedder: False
  Min semantic: 0.2
  Min LLM: 0.5
  LLM batch size: 15
  Hierarchical fallback: True


In [2]:
# =============================================================================
# SETUP: API KEY
# =============================================================================

os.environ["ANTHROPIC_API_KEY"] = ""

print("✓ API key configured")

✓ API key configured


## Step 1: Load Pre-built Graphs

Loads organizational and capability graphs without reprocessing.

In [3]:
# =============================================================================
# LOAD GRAPHS
# =============================================================================

import json
import networkx as nx
from bipartite_matcher import BipartiteCapabilityMatcher, MatcherConfig, MockEmbedder

org_graph = None
cap_graph = None

if LOAD_FROM == "local":
    print("Loading graphs from local JSON files...")
    
    # Load org graph
    if os.path.exists(ORG_GRAPH_PATH):
        with open(ORG_GRAPH_PATH, 'r', encoding='utf-8') as f:
            org_data = json.load(f)
        
        org_graph = nx.DiGraph()
        for node in org_data.get("nodes", []):
            node_id = node.pop("id")
            org_graph.add_node(node_id, **node)
        for edge in org_data.get("edges", []):
            org_graph.add_edge(edge["source"], edge["target"])
        
        print(f"  ✓ Org graph: {org_graph.number_of_nodes()} nodes, {org_graph.number_of_edges()} edges")
    else:
        print(f"  ⚠ Org graph not found: {ORG_GRAPH_PATH}")
    
    # Load cap graph
    if os.path.exists(CAP_GRAPH_PATH):
        with open(CAP_GRAPH_PATH, 'r', encoding='utf-8') as f:
            cap_data = json.load(f)
        
        cap_graph = nx.DiGraph()
        for node in cap_data.get("nodes", []):
            node_id = node.pop("id")
            cap_graph.add_node(node_id, **node)
        for edge in cap_data.get("edges", []):
            cap_graph.add_edge(edge["source"], edge["target"])
        
        print(f"  ✓ Cap graph: {cap_graph.number_of_nodes()} nodes, {cap_graph.number_of_edges()} edges")
    else:
        print(f"  ⚠ Cap graph not found: {CAP_GRAPH_PATH}")

elif LOAD_FROM == "falkordb":
    print("Loading graphs from FalkorDB...")
    
    from falkordb import FalkorDB
    client = FalkorDB.from_url(FALKORDB_URL_BASE)
    
    # Load org graph
    org_db = client.select_graph(ORG_GRAPH_NAME)
    org_graph = nx.DiGraph()
    
    result = org_db.query("MATCH (n) RETURN n")
    for record in result.result_set:
        node = record[0]
        props = dict(node.properties)
        node_id = props.pop("node_id", str(node.id))
        org_graph.add_node(node_id, **props)
    
    result = org_db.query("MATCH (a)-[r]->(b) RETURN a.node_id, b.node_id")
    for record in result.result_set:
        if record[0] and record[1]:
            org_graph.add_edge(record[0], record[1])
    
    print(f"  ✓ Org graph: {org_graph.number_of_nodes()} nodes")
    
    # Load cap graph
    cap_db = client.select_graph(CAP_GRAPH_NAME)
    cap_graph = nx.DiGraph()
    
    result = cap_db.query("MATCH (n) RETURN n")
    for record in result.result_set:
        node = record[0]
        props = dict(node.properties)
        node_id = props.pop("node_id", str(node.id))
        cap_graph.add_node(node_id, **props)
    
    result = cap_db.query("MATCH (a)-[r]->(b) RETURN a.node_id, b.node_id")
    for record in result.result_set:
        if record[0] and record[1]:
            cap_graph.add_edge(record[0], record[1])
    
    print(f"  ✓ Cap graph: {cap_graph.number_of_nodes()} nodes")

if org_graph and cap_graph:
    print("\n✓ Both graphs loaded successfully")
else:
    print("\n⚠ Failed to load graphs")

Loading graphs from local JSON files...
  ✓ Org graph: 529 nodes, 495 edges
  ✓ Cap graph: 383 nodes, 374 edges

✓ Both graphs loaded successfully


## Step 2: Filter Selected Organizations

In [4]:
# =============================================================================
# FILTER SELECTED ORGANIZATIONS
# =============================================================================

def get_org_leaves(graph):
    """Get leaf nodes (no children)."""
    return [n for n in graph.nodes() if graph.out_degree(n) == 0]

def filter_orgs_by_prefix(graph, prefixes):
    """Filter org nodes by code prefix."""
    if not prefixes:
        return list(graph.nodes())
    
    selected = []
    for node_id in graph.nodes():
        for prefix in prefixes:
            if node_id.startswith(prefix):
                selected.append(node_id)
                break
    return selected

# Get all org leaves
all_leaves = get_org_leaves(org_graph)
print(f"Total org leaves: {len(all_leaves)}")

# Filter by selected prefixes
if SELECTED_ORG_PREFIXES:
    # Get all nodes matching prefixes (including non-leaves for context)
    selected_nodes = filter_orgs_by_prefix(org_graph, SELECTED_ORG_PREFIXES)
    
    # Get leaves from selected
    selected_leaves = [n for n in selected_nodes if n in all_leaves]
    
    print(f"\nSelected prefixes: {SELECTED_ORG_PREFIXES}")
    print(f"Selected nodes: {len(selected_nodes)}")
    print(f"Selected leaves (for matching): {len(selected_leaves)}")
    
    # Show selected orgs
    print("\nSelected organizations:")
    for node_id in sorted(selected_leaves)[:10]:
        data = org_graph.nodes[node_id]
        print(f"  [{node_id}] {data.get('name', 'N/A')[:50]}")
    if len(selected_leaves) > 10:
        print(f"  ... and {len(selected_leaves) - 10} more")
else:
    selected_leaves = all_leaves
    print("\nNo filter applied - will match ALL org leaves")

target_orgs = selected_leaves

Total org leaves: 454

Selected prefixes: ['10']
Selected nodes: 71
Selected leaves (for matching): 65

Selected organizations:
  [10-10] 10-10
  [10-20] 10-20
  [10-2010] 10-2010
  [10A10] 10A10
  [10A1010] 10A1010
  [10A1020] 10A1020
  [10A20] 10A20
  [10A2010] 10A2010
  [10A30] 10A30
  [10A3010] 10A3010
  ... and 55 more


## Step 3: Setup Matcher & Run Matching

In [5]:
# =============================================================================
# SETUP LLM
# =============================================================================

from llm import create_llm

llm = create_llm(use_mock=USE_MOCK_LLM, model=LLM_MODEL)

C:\Users\marci\anaconda3\envs\DATAENLIGHT_AI_ARCH_PATTERNS\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ LLM: claude-haiku-4-5-20251001


In [6]:
# =============================================================================
# CREATE MATCHER WITH SELECTED MODE
# =============================================================================

from bipartite_matcher import (
    create_bipartite_matcher,
    BipartiteCapabilityMatcher,
    MatcherConfig,
    MockEmbedder,
    HuggingFaceEmbedder
)

# Create embedder
if USE_MOCK_EMBEDDER:
    embedder = MockEmbedder()
    print("Using MockEmbedder (hash-based)")
else:
    try:
        embedder = HuggingFaceEmbedder()
    except ImportError:
        print("⚠ sentence-transformers not available, using MockEmbedder")
        embedder = MockEmbedder()

# Create config
config = MatcherConfig(
    matching_mode=MATCHING_MODE,
    min_semantic_score=MIN_SEMANTIC_SCORE,
    min_llm_score=MIN_LLM_SCORE,
    top_k_candidates=TOP_K_CANDIDATES,
    llm_batch_size=LLM_BATCH_SIZE,
    use_llm_judge=True,
    enable_hierarchical_fallback=ENABLE_HIERARCHICAL_FALLBACK,
    sync_to_falkordb=USE_REAL_FALKORDB,
    verbose=True
)

# Create matcher
matcher = BipartiteCapabilityMatcher(org_graph, cap_graph, llm, embedder, config)

print(f"\n✓ Matcher created")
print(f"  Mode: {MATCHING_MODE}")
print(f"  Org nodes: {matcher.org_graph.number_of_nodes()}")
print(f"  Cap nodes: {matcher.cap_graph.number_of_nodes()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8519.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedder: all-MiniLM-L6-v2

✓ Matcher created
  Mode: llm_only
  Org nodes: 529
  Cap nodes: 383


In [7]:
# =============================================================================
# RUN MATCHING
# =============================================================================

match_results = []

if matcher:
    print("\n" + "═" * 70)
    print(f"RUNNING {MATCHING_MODE.upper()} MATCHING")
    print("═" * 70)
    
    # Use the unified run() method with target orgs
    match_results = matcher.run(target_org_ids=target_orgs)
    
    # Print summary
    matcher.print_summary()
    
    # Save results
    matcher.save_results(JSON_RESULTS_PATH)
    matcher.save_bipartite_graph("./graph_data/bipartite_matches.json")
    
    # Sync to FalkorDB if available
    if USE_REAL_FALKORDB:
        try:
            from falkordb import FalkorDB
            client = FalkorDB.from_url(FALKORDB_URL_BASE)
            matcher.sync_to_falkordb(client)
        except Exception as e:
            print(f"⚠ FalkorDB sync failed: {e}")
else:
    print("⏭️ Matching skipped (no matcher)")


══════════════════════════════════════════════════════════════════════
RUNNING LLM_ONLY MATCHING
══════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════
MATCHING MODE: LLM_ONLY
══════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════
LLM-ONLY CAPABILITY MATCHING
══════════════════════════════════════════════════════════════════════
Mode: Full LLM evaluation (no semantic pre-filtering)

Org leaves to match: 65
Cap leaves to search: 328
Estimated LLM calls: ~1430

--- LLM-Only Matching ---

[1/65] Evaluating: 10-10 - 10-10
  → Monitor
    Score: 0.92 (MODERATE)

[2/65] Evaluating: 10-20 - 10-20
  → Governance Structures
    Score: 0.95 (MODERATE)

[3/65] Evaluating: 10-2010 - 10-2010
  → Security incident management
    Score: 0.95 (MODERATE)

[4/65] Evaluating: 10A10 - 10A10
  → Understanding Needs
    Score: 0.92 (

## Step 4: Store Results & Vectorize in Graph

In [8]:
# =============================================================================
# STORE RESULTS IN GRAPHS
# =============================================================================

def store_matches_in_graphs(matcher, results, org_graph, cap_graph):
    """Store match results and embeddings in both NetworkX and FalkorDB."""
    
    print("\nStoring matches in graphs...")
    
    # Update org nodes with their matches
    for result in results:
        if result.org_id in org_graph.nodes():
            best = result.best_match()
            if best:
                org_graph.nodes[result.org_id]["matched_capability"] = best["capability_id"]
                org_graph.nodes[result.org_id]["match_score"] = best["combined_score"]
                org_graph.nodes[result.org_id]["match_type"] = best["match_type"]
                org_graph.nodes[result.org_id]["match_justification"] = best["justification"]
    
    print(f"  ✓ Updated {len(results)} org nodes in NetworkX")
    
    # Sync to FalkorDB
    if USE_REAL_FALKORDB:
        try:
            from falkordb import FalkorDB
            client = FalkorDB.from_url(FALKORDB_URL_BASE)
            
            # Sync bipartite matches
            count = matcher.sync_to_falkordb(client)
            
            # Update org nodes with match info
            org_db = client.select_graph(ORG_GRAPH_NAME)
            for result in results:
                best = result.best_match()
                if best:
                    query = """
                    MATCH (n {node_id: $node_id})
                    SET n.matched_capability = $cap_id,
                        n.match_score = $score,
                        n.match_type = $match_type,
                        n.match_justification = $justification
                    """
                    org_db.query(query, {
                        "node_id": result.org_id,
                        "cap_id": best["capability_id"],
                        "score": best["combined_score"],
                        "match_type": best["match_type"],
                        "justification": best["justification"]
                    })
            
            print(f"  ✓ Synced to FalkorDB")
        except Exception as e:
            print(f"  ⚠ FalkorDB sync failed: {e}")
    
    return len(results)

store_matches_in_graphs(matcher, match_results, org_graph, cap_graph)


Storing matches in graphs...
  ✓ Updated 65 org nodes in NetworkX

Syncing matches to FalkorDB...
✓ Synced 865 matches to FalkorDB
  ✓ Synced to FalkorDB


65

In [9]:
# =============================================================================
# VECTORIZE MATCH EMBEDDINGS
# =============================================================================

def vectorize_matches(matcher, org_graph):
    """Store embeddings for matched orgs."""
    
    print("\nVectorizing org embeddings...")
    
    count = 0
    for org_id, embedding in matcher._org_embeddings.items():
        if org_id in org_graph.nodes():
            org_graph.nodes[org_id]["embedding"] = embedding
            count += 1
    
    print(f"  ✓ Stored {count} embeddings in NetworkX")
    
    # Sync to FalkorDB
    if USE_REAL_FALKORDB:
        try:
            from falkordb import FalkorDB
            client = FalkorDB.from_url(FALKORDB_URL_BASE)
            org_db = client.select_graph(ORG_GRAPH_NAME)
            
            synced = 0
            for org_id, embedding in matcher._org_embeddings.items():
                try:
                    query = """
                    MATCH (n {node_id: $node_id})
                    SET n.embedding = $embedding
                    """
                    org_db.query(query, {"node_id": org_id, "embedding": embedding})
                    synced += 1
                except:
                    pass
            
            print(f"  ✓ Synced {synced} embeddings to FalkorDB")
        except Exception as e:
            print(f"  ⚠ FalkorDB sync failed: {e}")

vectorize_matches(matcher, org_graph)


Vectorizing org embeddings...
  ✓ Stored 0 embeddings in NetworkX
  ✓ Synced 0 embeddings to FalkorDB


## Step 5: Generate Excel Report

In [10]:
# =============================================================================
# GENERATE EXCEL REPORT
# =============================================================================

import pandas as pd
from pathlib import Path

def get_org_hierarchy_path(org_graph, org_id):
    """Get full hierarchy path for an organization (root to leaf)."""
    path_codes = [org_id]
    path_names = [org_graph.nodes[org_id].get("name", org_id)]
    current = org_id
    
    while True:
        preds = list(org_graph.predecessors(current))
        if not preds:
            break
        parent = preds[0]
        parent_data = org_graph.nodes.get(parent, {})
        parent_name = parent_data.get("name", parent)
        # Skip master root
        if "MASTER" not in parent.upper() and "ROOT" not in parent.upper():
            path_codes.insert(0, parent)
            path_names.insert(0, parent_name)
        current = parent
    
    return {
        "codes": " > ".join(path_codes),
        "names": " > ".join(path_names)
    }

def get_capability_hierarchy(cap_graph, cap_id):
    """Get full capability hierarchy with codes, names, and descriptions at each level."""
    hierarchy = {
        "l1_code": "", "l1_name": "", "l1_desc": "",
        "l2_code": "", "l2_name": "", "l2_desc": "",
        "l3_code": "", "l3_name": "", "l3_desc": "",
        "full_path": ""
    }
    
    # Build path from leaf to root
    path = []
    current = cap_id
    while current:
        data = cap_graph.nodes.get(current, {})
        if "ROOT" not in current.upper():
            path.insert(0, {
                "code": current,
                "name": data.get("name", current),
                "desc": data.get("refined_description", data.get("description", ""))
            })
        preds = list(cap_graph.predecessors(current))
        current = preds[0] if preds else None
    
    # Assign to levels
    for i, item in enumerate(path):
        if i == 0:
            hierarchy["l1_code"] = item["code"]
            hierarchy["l1_name"] = item["name"]
            hierarchy["l1_desc"] = item["desc"]
        elif i == 1:
            hierarchy["l2_code"] = item["code"]
            hierarchy["l2_name"] = item["name"]
            hierarchy["l2_desc"] = item["desc"]
        elif i == 2:
            hierarchy["l3_code"] = item["code"]
            hierarchy["l3_name"] = item["name"]
            hierarchy["l3_desc"] = item["desc"]
    
    hierarchy["full_path"] = " > ".join([p["name"] for p in path])
    return hierarchy

def generate_excel_report(results, org_graph, cap_graph, output_path):
    """Generate detailed Excel report with organization and capability hierarchy details."""
    
    print("\nGenerating Excel report...")
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # ═══════════════════════════════════════════════════════════════════════
    # Sheet 1: Match Summary (enhanced)
    # ═══════════════════════════════════════════════════════════════════════
    summary_rows = []
    for result in results:
        org_data = org_graph.nodes.get(result.org_id, {})
        org_path = get_org_hierarchy_path(org_graph, result.org_id)
        best = result.best_match()
        
        row = {
            # Organization details
            "Org Code": result.org_id,
            "Org Name": result.org_name,
            "Org Level": result.org_level,
            "Org Hierarchy (Codes)": org_path["codes"],
            "Org Hierarchy (Names)": org_path["names"],
            "Org Description": org_data.get("refined_description", "")[:300],
            
            # Match status
            "Status": "MATCHED" if not result.unmatched else "UNMATCHED",
            "Fallback Level": result.fallback_level,
            "# Matches": len(result.matches),
        }
        
        if best:
            cap_hier = get_capability_hierarchy(cap_graph, best["capability_id"])
            row.update({
                # Capability hierarchy
                "Cap L1 Code": cap_hier["l1_code"],
                "Cap L1 Name": cap_hier["l1_name"],
                "Cap L2 Code": cap_hier["l2_code"],
                "Cap L2 Name": cap_hier["l2_name"],
                "Cap L3 Code": cap_hier["l3_code"],
                "Cap L3 Name": cap_hier["l3_name"],
                "Cap Full Path": cap_hier["full_path"],
                
                # Scores
                "Semantic Score": best["semantic_score"],
                "LLM Score": best["llm_score"],
                "Combined Score": best["combined_score"],
                "Match Type": best["match_type"],
                
                # Justification
                "Justification": best["justification"],
                "Key Overlaps": best.get("key_overlaps", ""),
                "Gaps": best.get("gaps", "")
            })
        
        summary_rows.append(row)
    
    df_summary = pd.DataFrame(summary_rows)
    
    # ═══════════════════════════════════════════════════════════════════════
    # Sheet 2: All Matches Detail (enhanced)
    # ═══════════════════════════════════════════════════════════════════════
    detail_rows = []
    for result in results:
        org_data = org_graph.nodes.get(result.org_id, {})
        org_path = get_org_hierarchy_path(org_graph, result.org_id)
        
        # Get org activities formatted
        activities = org_data.get("activities", [])
        weights = org_data.get("activity_weights", [])
        activities_formatted = "; ".join([
            f"({w}%) {a[:100]}" for a, w in zip(activities, weights)
        ]) if activities else ""
        
        for match in result.matches:
            cap_data = cap_graph.nodes.get(match["capability_id"], {})
            cap_hier = get_capability_hierarchy(cap_graph, match["capability_id"])
            
            row = {
                # Organization full details
                "Org Code": result.org_id,
                "Org Name": result.org_name,
                "Org Level": result.org_level,
                "Org Hierarchy (Codes)": org_path["codes"],
                "Org Hierarchy (Names)": org_path["names"],
                "Org Activities": activities_formatted[:500],
                "Org Refined Description": org_data.get("refined_description", ""),
                "Org Activity Summary": org_data.get("refined_activity_summary", ""),
                
                # Capability full details
                "Cap Code": match["capability_id"],
                "Cap Name": match["capability_name"],
                "Cap Level": match["capability_level"],
                "Cap Type": cap_data.get("node_type", ""),
                
                # Capability hierarchy
                "Cap L1 Code": cap_hier["l1_code"],
                "Cap L1 Name": cap_hier["l1_name"],
                "Cap L1 Description": cap_hier["l1_desc"][:300],
                "Cap L2 Code": cap_hier["l2_code"],
                "Cap L2 Name": cap_hier["l2_name"],
                "Cap L2 Description": cap_hier["l2_desc"][:300],
                "Cap L3 Code": cap_hier["l3_code"],
                "Cap L3 Name": cap_hier["l3_name"],
                "Cap L3 Description": cap_hier["l3_desc"][:300],
                "Cap Full Path": cap_hier["full_path"],
                "Cap Keywords": cap_data.get("capability_keywords", ""),
                
                # Match scores
                "Semantic Score": match["semantic_score"],
                "LLM Score": match["llm_score"],
                "Combined Score": match["combined_score"],
                "Match Type": match["match_type"],
                
                # Match justification
                "Justification": match["justification"],
                "Key Overlaps": match.get("key_overlaps", ""),
                "Gaps": match.get("gaps", ""),
                
                # Fallback info
                "Fallback Level": result.fallback_level
            }
            detail_rows.append(row)
    
    df_details = pd.DataFrame(detail_rows)
    
    # ═══════════════════════════════════════════════════════════════════════
    # Sheet 3: Unmatched Orgs (enhanced)
    # ═══════════════════════════════════════════════════════════════════════
    unmatched_rows = []
    for result in results:
        if result.unmatched:
            org_data = org_graph.nodes.get(result.org_id, {})
            org_path = get_org_hierarchy_path(org_graph, result.org_id)
            
            activities = org_data.get("activities", [])
            weights = org_data.get("activity_weights", [])
            activities_formatted = "; ".join([
                f"({w}%) {a}" for a, w in zip(activities, weights)
            ]) if activities else ""
            
            row = {
                "Org Code": result.org_id,
                "Org Name": result.org_name,
                "Org Level": result.org_level,
                "Org Hierarchy (Codes)": org_path["codes"],
                "Org Hierarchy (Names)": org_path["names"],
                "Activities": activities_formatted,
                "Refined Description": org_data.get("refined_description", ""),
                "Activity Summary": org_data.get("refined_activity_summary", "")
            }
            unmatched_rows.append(row)
    
    df_unmatched = pd.DataFrame(unmatched_rows)
    
    # ═══════════════════════════════════════════════════════════════════════
    # Sheet 4: Statistics
    # ═══════════════════════════════════════════════════════════════════════
    matched = [r for r in results if not r.unmatched]
    all_scores = [r.best_match()["combined_score"] for r in matched if r.best_match()]
    sem_scores = [r.best_match()["semantic_score"] for r in matched if r.best_match()]
    llm_scores = [r.best_match()["llm_score"] for r in matched if r.best_match()]
    
    type_counts = {"STRONG": 0, "MODERATE": 0, "WEAK": 0}
    fallback_counts = {0: 0, 1: 0, 2: 0}
    for r in matched:
        best = r.best_match()
        if best:
            t = best.get("match_type", "UNKNOWN")
            type_counts[t] = type_counts.get(t, 0) + 1
        fb = r.fallback_level
        fallback_counts[min(fb, 2)] = fallback_counts.get(min(fb, 2), 0) + 1
    
    stats = {
        "Metric": [
            "Total Orgs Processed",
            "Matched",
            "Unmatched",
            "Match Rate",
            "Avg Semantic Score",
            "Avg LLM Score",
            "Avg Combined Score",
            "Strong Matches",
            "Moderate Matches",
            "Weak Matches",
            "Fallback Level 0 (Leaf)",
            "Fallback Level 1",
            "Fallback Level 2+"
        ],
        "Value": [
            len(results),
            len(matched),
            len(results) - len(matched),
            f"{len(matched)/len(results)*100:.1f}%" if results else "0%",
            f"{sum(sem_scores)/len(sem_scores):.3f}" if sem_scores else "N/A",
            f"{sum(llm_scores)/len(llm_scores):.3f}" if llm_scores else "N/A",
            f"{sum(all_scores)/len(all_scores):.3f}" if all_scores else "N/A",
            type_counts.get("STRONG", 0),
            type_counts.get("MODERATE", 0),
            type_counts.get("WEAK", 0),
            fallback_counts.get(0, 0),
            fallback_counts.get(1, 0),
            fallback_counts.get(2, 0)
        ]
    }
    
    df_stats = pd.DataFrame(stats)
    
    # ═══════════════════════════════════════════════════════════════════════
    # Sheet 5: Capability Hierarchy Reference
    # ═══════════════════════════════════════════════════════════════════════
    cap_ref_rows = []
    matched_cap_ids = set()
    for result in results:
        for match in result.matches:
            matched_cap_ids.add(match["capability_id"])
    
    for cap_id in sorted(matched_cap_ids):
        cap_data = cap_graph.nodes.get(cap_id, {})
        cap_hier = get_capability_hierarchy(cap_graph, cap_id)
        
        row = {
            "Cap Code": cap_id,
            "Cap Name": cap_data.get("name", ""),
            "Cap Level": cap_data.get("level", ""),
            "Cap Type": cap_data.get("node_type", ""),
            "L1 Code": cap_hier["l1_code"],
            "L1 Name": cap_hier["l1_name"],
            "L1 Description": cap_hier["l1_desc"],
            "L2 Code": cap_hier["l2_code"],
            "L2 Name": cap_hier["l2_name"],
            "L2 Description": cap_hier["l2_desc"],
            "L3 Code": cap_hier["l3_code"],
            "L3 Name": cap_hier["l3_name"],
            "L3 Description": cap_hier["l3_desc"],
            "Keywords": cap_data.get("capability_keywords", ""),
            "Full Path": cap_hier["full_path"]
        }
        cap_ref_rows.append(row)
    
    df_cap_ref = pd.DataFrame(cap_ref_rows)
    
    # ═══════════════════════════════════════════════════════════════════════
    # Write to Excel
    # ═══════════════════════════════════════════════════════════════════════
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Match Summary', index=False)
        df_details.to_excel(writer, sheet_name='All Matches Detail', index=False)
        df_unmatched.to_excel(writer, sheet_name='Unmatched Orgs', index=False)
        df_stats.to_excel(writer, sheet_name='Statistics', index=False)
        df_cap_ref.to_excel(writer, sheet_name='Capability Reference', index=False)
    
    print(f"✓ Excel report saved: {output_path}")
    print(f"  Sheets:")
    print(f"    - Match Summary: {len(df_summary)} rows")
    print(f"    - All Matches Detail: {len(df_details)} rows")
    print(f"    - Unmatched Orgs: {len(df_unmatched)} rows")
    print(f"    - Statistics: key metrics")
    print(f"    - Capability Reference: {len(df_cap_ref)} unique capabilities")
    
    return output_path

generate_excel_report(match_results, org_graph, cap_graph, EXCEL_REPORT_PATH)


Generating Excel report...
✓ Excel report saved: ./output/capability_mapping_report.xlsx
  Sheets:
    - Match Summary: 65 rows
    - All Matches Detail: 865 rows
    - Unmatched Orgs: 0 rows
    - Statistics: key metrics
    - Capability Reference: 235 unique capabilities


'./output/capability_mapping_report.xlsx'

## Step 6: Save JSON Results

In [11]:
# =============================================================================
# SAVE JSON RESULTS
# =============================================================================

def save_json_results(results, output_path):
    """Save match results to JSON."""
    
    output = {
        "metadata": {
            "selected_prefixes": SELECTED_ORG_PREFIXES,
            "total_orgs": len(results),
            "matched": sum(1 for r in results if not r.unmatched),
            "unmatched": sum(1 for r in results if r.unmatched),
            "config": {
                "use_llm_judge": USE_LLM_JUDGE,
                "min_semantic_score": MIN_SEMANTIC_SCORE,
                "min_llm_score": MIN_LLM_SCORE,
                "hierarchical_fallback": ENABLE_HIERARCHICAL_FALLBACK
            }
        },
        "matches": [r.to_dict() for r in results]
    }
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    
    print(f"\n✓ JSON results saved: {output_path}")

save_json_results(match_results, JSON_RESULTS_PATH)

# Also save bipartite graph
matcher.save_bipartite_graph("./graph_data/selected_bipartite_matches.json")

NameError: name 'USE_LLM_JUDGE' is not defined

## Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "═" * 70)
print("CAPABILITY MAPPING SUMMARY")
print("═" * 70)

matched = [r for r in match_results if not r.unmatched]
unmatched = [r for r in match_results if r.unmatched]

print(f"\nSelected orgs: {SELECTED_ORG_PREFIXES or 'ALL'}")
print(f"Total processed: {len(match_results)}")
print(f"Matched: {len(matched)} ({len(matched)/len(match_results)*100:.1f}%)" if match_results else "N/A")
print(f"Unmatched: {len(unmatched)}")

if matched:
    type_counts = {}
    for r in matched:
        best = r.best_match()
        if best:
            t = best["match_type"]
            type_counts[t] = type_counts.get(t, 0) + 1
    
    print("\nMatch types:")
    for t, c in sorted(type_counts.items()):
        print(f"  {t}: {c}")

print(f"\nOutputs:")
print(f"  Excel: {EXCEL_REPORT_PATH}")
print(f"  JSON: {JSON_RESULTS_PATH}")
print(f"  FalkorDB: {MATCH_GRAPH_NAME if USE_REAL_FALKORDB else 'N/A'}")

print("\n" + "═" * 70)